In [1]:
import pandas as pd
import json
import numpy as np
from collections import defaultdict
import contextlib
from io import StringIO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [2]:
# READ CSV
firstCSV = pd.read_csv("PathsFromANON7TDVLK188_1_6.csv", header=None)
secondCSV = pd.read_csv("PathsFromANONAVDVLK13R_1_4.csv", header=None)
firstCSV = firstCSV[0].str.split('/').str[1]
secondCSV = secondCSV[0].str.split('/').str[1]

In [3]:
def load_pred_by_patch(pred_json_path, keys_keep=None):
    with open(pred_json_path, "r") as f:
        preds = json.load(f)

    pred_by_key = defaultdict(list)
    for det in preds:
        key = det.get("image_id")
        if key is None:
            continue
        if keys_keep is not None and key not in keys_keep:
            continue

        x, y, w, h = det["bbox"]        # already pixel xywh
        score = float(det["score"])
        pred_by_key[key].append([float(x), float(y), float(w), float(h), score])

    return dict(pred_by_key)

In [4]:
YOLOv10nData = load_pred_by_patch("predictions_YOLOv10n.json")
YOLOv11nData = load_pred_by_patch("predictions_YOLOv11n.json")
YOLOv12nData = load_pred_by_patch("predictions_YOLOv12n.json")

In [5]:
def load_ground_truth_by_patch(dataset):
    ground_truth_by_key = defaultdict(list)

    for patch in dataset:
        with open(f'Dataset/labels/test/{patch}.txt', "r") as f:
            for line in f:
                parts = line.split()
                if(len(parts) < 5):
                    continue

                center_x_non_normalized = float(parts[1])*640
                center_y_non_normalized = float(parts[2])*640
                width_non_normalized = float(parts[3])*640
                height_non_normalized = float(parts[4])*640

                top_left_x = center_x_non_normalized - (width_non_normalized / 2)
                if(top_left_x < 0):
                    top_left_x = 0

                top_left_y = center_y_non_normalized - (height_non_normalized / 2)

                if(top_left_y < 0):
                    top_left_y = 0
                    
                ground_truth_by_key[patch].append([top_left_x, top_left_y, width_non_normalized, height_non_normalized])
    return dict(ground_truth_by_key)

In [6]:
concat_labels = pd.concat([firstCSV, secondCSV], ignore_index=True)
ground_truth = load_ground_truth_by_patch(concat_labels)

In [7]:
def iou(groundTruth, prediction):
    # convert from center-format to corner-format
    bbox1_top_left_x = groundTruth[0] - groundTruth[2]/2
    bbox1_top_left_y = groundTruth[1] - groundTruth[3]/2
    bbox1_bottom_right_x = bbox1_top_left_x + groundTruth[2]
    bbox1_bottom_right_y = bbox1_top_left_y + groundTruth[3]

    bbox1_top_left_x = groundTruth[0]
    bbox1_top_left_y = groundTruth[1]
    bbox1_bottom_right_x = bbox1_top_left_x + groundTruth[2]
    bbox1_bottom_right_y = bbox1_top_left_y + groundTruth[3]
    #print("BBOX1")
    #print(f'Top left: x = {bbox1_top_left_x}, y = {bbox1_top_left_y}')
    #print(f'Bottom right: x = {bbox1_bottom_right_x}, y = {bbox1_bottom_right_y}')

    #print("\n\nBBOX2")
    #bbox2_top_left_x = bbox2[0] - bbox2[2]/2
    #bbox2_top_left_y = bbox2[1] - bbox2[3]/2
    #bbox2_bottom_right_x = bbox2_top_left_x + bbox2[2]
    #bbox2_bottom_right_y = bbox2_top_left_y + bbox2[3]
    # BOX 2 HAS TOP LEFT FORMAT ALREADY
    bbox2_top_left_x = prediction[0]
    bbox2_top_left_y = prediction[1]
    bbox2_bottom_right_x = bbox2_top_left_x + prediction[2]
    bbox2_bottom_right_y = bbox2_top_left_y + prediction[3]
    #print(f'Top left: x = {bbox2_top_left_x}, y = {bbox2_top_left_y}')
    #print(f'Bottom right: x = {bbox2_bottom_right_x}, y = {bbox2_bottom_right_y}')

    # Intersection
    x_intersection_1 = max(bbox1_top_left_x, bbox2_top_left_x)
    y_intersection_1 = max(bbox1_top_left_y, bbox2_top_left_y)

    x_intersection_2 = min(bbox1_bottom_right_x, bbox2_bottom_right_x)
    y_intersection_2 = min(bbox1_bottom_right_y, bbox2_bottom_right_y)

    inter_w = max(0, x_intersection_2 - x_intersection_1)
    inter_h = max(0, y_intersection_2 - y_intersection_1)
    inter_area = inter_w * inter_h
    #print(f'\n\nInter area: {inter_area}')
    # UNION
    area_bbox1 = groundTruth[2] * groundTruth[3]
    area_bbox2 = prediction[2] * prediction[3]
    union = area_bbox1 + area_bbox2 - inter_area

    #print("IOU: ", inter_area/union)
    #IoU
    return inter_area/union

In [8]:
def calculateMetricsOverSample(key, model):    
    iou_thr = 0.5
    conf_thr = 0.34

    tp = 0
    fp = 0
    fn = 0

    predictions = model.get(key) #find all predictions for sample patch
    groundTruthForSample = ground_truth.get(key)

    # If there are no predictions, return 0 fp, 0 tp and fn as length of ground truth boxes
    if not predictions:
        return tp, fp, len(groundTruthForSample)
    
    matched_gt = [False] * len(groundTruthForSample)

    #print(f'\nIteration {counter}')
    for p in predictions:
        #print("Prediction: ", p)
        #p_box = p[:4]  # drop score for IoU computation

        if(p[4] < conf_thr):
                #print(f'Skipping as conf is {p[4]} .......')
            continue
        best_iou = 0.0
        best_j = -1
    
        # Find best unmatched GT for this prediction
        for j, gt_box in enumerate(groundTruthForSample):
            #print("Ground truth: ", gt_box)
            if matched_gt[j]:
                continue
    
            iou_val = iou(gt_box, p)
            if iou_val > best_iou:
                best_iou = iou_val
                best_j = j
    
        # Decide TP vs FP
        if best_iou >= iou_thr and best_j != -1:
            tp += 1
            matched_gt[best_j] = True
        else:
            fp += 1
    
    # Unmatched GTs are false negatives
    fn += matched_gt.count(False)

    return tp, fp, fn

In [9]:
# CODE CREATED WITH ASSISTANCE FROM CHATGPT
def create_coco_ground_truth(combinedSamples):
    """
    Convert ground truth dictionary to COCO format
    
    Args:
        gt_dict: Dictionary with {image_name: [[top_left_x, top_left_y, width, height], ...]}
    
    Returns:
        Dictionary in COCO format
    """
    coco_gt = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 1, "name": "object"}]
    }
    
    annotation_id = 1
    
    for image_id, image_name in enumerate(combinedSamples, start=1):
        # Add image info
        coco_gt["images"].append({
            "id": image_id,
            "file_name": image_name,
            "width": 640,
            "height": 640
        })
        
        # Add annotations for this image
        for bbox in ground_truth[image_name]:
            x_min, y_min, width, height = bbox
            
            coco_gt["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [x_min, y_min, width, height],
                "area": width * height,
                "iscrowd": 0
            })
            annotation_id += 1
    
    return coco_gt

In [10]:
# CODE CREATED WITH ASSISTANCE FROM CHATGPT
def create_coco_predictions(pred_dict, combinedSamples):
    """
    Convert predictions dictionary to COCO format
    
    Args:
        pred_dict: Dictionary with {image_name: [[top_left_x, top_left_y, width, height, conf], ...]}
        image_name_to_id: Dictionary mapping image names to COCO image IDs
    
    Returns:
        List of predictions in COCO format
    """
    coco_predictions = []

    for image_id, image_name in enumerate(combinedSamples, start=1):
        if(image_name in pred_dict):
            predictions = pred_dict[f"{image_name}"]
            for pred in predictions:
                x_min, y_min, width, height, confidence = pred

                coco_predictions.append({
                        "image_id": image_id,
                        "category_id": 1,
                        "bbox": [x_min, y_min, width, height],
                        "score": confidence
                    })
    
    return coco_predictions

In [11]:
# METHOD FOR STATISTICAL BOOTSTRAPPING
models = [YOLOv10nData, YOLOv11nData, YOLOv12nData]
modelNames = ["YOLOv10nData", "YOLOv11nData", "YOLOv12nData"]
modelStats = defaultdict(list)
bootstrap = 5#1000 -> Increase to 1000, it's very time consuming -> results are already available in "boostrapResults.json"
seed_value_for_random = 42
rng = np.random.default_rng(seed=seed_value_for_random)

for iteration in range(bootstrap):
    print(f'{iteration + 1}. Bootstrap iteration')
    # Sample images
    sample = rng.choice(firstCSV.to_numpy().flatten(), size=len(firstCSV), replace=True)
    sample2 = rng.choice(secondCSV.to_numpy().flatten(), size=len(secondCSV), replace=True)
    combinedSamples = np.concatenate((sample, sample2))
    
    for i in range(len(models)):
        #print(f'Processing {modelNames[i]}...')
        TP = 0
        FP = 0
        FN = 0
        
        for gt in combinedSamples:
            tp, fp, fn = calculateMetricsOverSample(gt, models[i])
            TP += tp
            FP += fp
            FN += fn
        
        precision = TP/(TP+FP)
        recall = TP/(TP+FN)

        coco_gt_dict = create_coco_ground_truth(combinedSamples)
        coco_predictions = create_coco_predictions(models[i], combinedSamples)

        with open('test_gt.json', 'w') as f:
            json.dump(coco_gt_dict, f)
            
        # CODE CREATED WITH ASSISTANCE FROM CHATGPT
        with contextlib.redirect_stdout(StringIO()):
            gt_coco = COCO('test_gt.json')
            dt_coco = gt_coco.loadRes(coco_predictions)
            coco_eval = COCOeval(gt_coco, dt_coco, 'bbox')
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()
        
        # Calculate AP using COCO eval for the bootstrap sample
        # You need gt_dict and pred_dict for this model
        #ap5095, ap50 = calculate_ap_for_sample(ground_truth, models[i], combinedSamples)
        
        modelStats[modelNames[i]].append([precision, recall, coco_eval.stats[0], coco_eval.stats[1]])
        #print(f'Writing results...\n')

1. Bootstrap iteration
2. Bootstrap iteration
3. Bootstrap iteration
4. Bootstrap iteration
5. Bootstrap iteration


In [12]:
# To save all bootstrap metrics to .json form

In [13]:
with open("bootstrapResult.json", "w") as f:
    json.dump(modelStats, f, indent=4)